# Spotify Hit Prediction - Analysis Notebook

**Question:** can we predict whether a track will be a *hit* (Spotify `popularity >= 50`) from its audio features and genre?

This notebook is the exploratory/modelling companion to the MLOps pipeline in this repo (it replaces the Telco churn notebook).
Mapping from the churn template: `Churn` -> `Hit`, `customerID` -> `track_id`, recall-based quality gate unchanged.

Run it from the `notebooks/` folder; it uses the files produced by `src/prepare_spotify_data.py` and pipeline stage `03_preprocessing.py`.

In [ ]:
import os, subprocess, sys, warnings
warnings.filterwarnings("ignore")
os.chdir("..") if os.path.basename(os.getcwd()) == "notebooks" else None

import numpy as np, pandas as pd, joblib, yaml
import matplotlib
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
                             precision_recall_curve)

cfg = yaml.safe_load(open("config.yaml"))
os.makedirs("artifacts", exist_ok=True)
print("working dir:", os.getcwd())

## 1. Data and target

In [ ]:
raw = pd.read_csv("data/raw/dataset.csv")          # original dump: one row per (track, genre)
df  = pd.read_csv(cfg["data"]["raw_path"])          # cleaned: one row per track_id, target `Hit` added
print("original rows :", len(raw), "| unique tracks:", raw.track_id.nunique())
print("cleaned rows  :", len(df))
print("Hit rate      : %.1f%%" % (100 * (df.Hit == "Yes").mean()))
df.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df.popularity.hist(bins=50, ax=ax[0]); ax[0].axvline(cfg["data"]["hit_threshold"], color="red", ls="--", label="hit threshold")
ax[0].set_title("Popularity distribution"); ax[0].legend()
df.Hit.value_counts().plot.bar(ax=ax[1], color=["#999", "#1db954"]); ax[1].set_title("Target: Hit")
plt.tight_layout(); plt.savefig("artifacts/target_distribution.png", dpi=120); plt.show()

**Things worth knowing about this dataset**
- The same track appears once per genre, so naive splitting would put copies of one song in both train and test (**leakage**). We keep one row per `track_id`.
- About 10% of tracks have `popularity == 0` (likely very new or obscure tracks) - these are all non-hits.
- `popularity` defines the target, so it must never be a feature.

## 2. Audio features vs. hit status

In [ ]:
num = ["danceability","energy","loudness","speechiness","acousticness","instrumentalness","liveness","valence","tempo","duration_ms"]
summary = df.groupby("Hit")[num].median().T
summary["diff_%"] = (100 * (summary["Yes"] - summary["No"]) / summary["No"].abs()).round(1)
summary

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for a, c in zip(axes.ravel(), num):
    lo, hi = df[c].quantile([0.01, 0.99])
    for lab, col in [("No", "#999"), ("Yes", "#1db954")]:
        df.loc[df.Hit == lab, c].clip(lo, hi).plot.hist(bins=30, alpha=.55, density=True, ax=a, color=col, label=lab)
    a.set_title(c); a.set_ylabel("")
axes[0, 0].legend(); plt.tight_layout(); plt.savefig("artifacts/feature_distributions.png", dpi=110); plt.show()

In [ ]:
corr = df[num + ["popularity"]].corr()["popularity"].drop("popularity").sort_values()
corr.plot.barh(figsize=(6, 4), title="Correlation with popularity"); plt.tight_layout(); plt.show()

## 3. Genre matters most

In [ ]:
g = df.groupby("track_genre").agg(tracks=("Hit", "size"), hit_rate=("Hit", lambda s: (s == "Yes").mean())).query("tracks >= 100")
top, bottom = g.sort_values("hit_rate").tail(10), g.sort_values("hit_rate").head(10)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
top.hit_rate.plot.barh(ax=ax[0], color="#1db954", title="Highest hit rate by genre")
bottom.hit_rate.plot.barh(ax=ax[1], color="#999", title="Lowest hit rate by genre")
plt.tight_layout(); plt.savefig("artifacts/genre_hit_rate.png", dpi=110); plt.show()

## 4. Preprocessing (same code as pipeline stage 03)

In [ ]:
subprocess.run([sys.executable, "src/mlops_pipeline/03_preprocessing.py"], check=True,
               env={**os.environ, "PYTHONPATH": "."}, capture_output=True)
d = cfg["data"]["processed_dir"]
Xtr, Xte = np.load(f"{d}/X_train_final.npy"), np.load(f"{d}/X_test_final.npy")
ytr, yte = np.load(f"{d}/y_train.npy"), np.load(f"{d}/y_test.npy")
pre = joblib.load(cfg["paths"]["preprocessor"])
names = pre.get_feature_names_out()
print("train", Xtr.shape, "| test", Xte.shape, "| positive rate train/test: %.3f / %.3f" % (ytr.mean(), yte.mean()))

## 5. Model comparison

In [ ]:
models = {
    "Dummy (always No)":      DummyClassifier(strategy="most_frequent"),
    "Logistic Regression":    LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree (d=8)":    DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=42),
    "Random Forest (config)": RandomForestClassifier(**{k: cfg["model"][k] for k in
                              ["n_estimators","max_depth","min_samples_leaf","random_state","class_weight","n_jobs"]}),
    "HistGradientBoosting":   HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
}
rows, fitted = [], {}
for name, m in models.items():
    m.fit(Xtr, ytr); fitted[name] = m
    p = m.predict_proba(Xte)[:, 1]; yhat = (p >= 0.5).astype(int)
    rows.append(dict(model=name, accuracy=accuracy_score(yte, yhat), precision=precision_score(yte, yhat, zero_division=0),
                     recall=recall_score(yte, yhat), f1=f1_score(yte, yhat), roc_auc=roc_auc_score(yte, p)))
results = pd.DataFrame(rows).set_index("model").round(4)
results

Accuracy looks *worse* than the dummy model on purpose: the models use `class_weight="balanced"` to favour **recall** on the minority class (hits), which is what the quality gate measures. ROC-AUC and recall are the meaningful columns here.

## 6. Chosen model: evaluation plots

In [ ]:
rf = fitted["Random Forest (config)"]; p = rf.predict_proba(Xte)[:, 1]; yhat = (p >= 0.5).astype(int)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(confusion_matrix(yte, yhat), display_labels=["Not hit", "Hit"]).plot(ax=ax[0], cmap="Greens", colorbar=False)
RocCurveDisplay.from_predictions(yte, p, ax=ax[1], name="RandomForest"); ax[1].plot([0, 1], [0, 1], "k--", lw=.8)
plt.tight_layout(); plt.savefig("artifacts/evaluation_plots.png", dpi=120); plt.show()
fig, a = plt.subplots(figsize=(4.5, 4)); ConfusionMatrixDisplay(confusion_matrix(yte, yhat), display_labels=["Not hit","Hit"]).plot(ax=a, cmap="Greens", colorbar=False)
fig.savefig("artifacts/confusion_matrix.png", dpi=120); plt.close(fig)
fig, a = plt.subplots(figsize=(4.5, 4)); RocCurveDisplay.from_predictions(yte, p, ax=a); fig.savefig("artifacts/roc_curve.png", dpi=120); plt.close(fig)
print("recall=%.3f precision=%.3f roc_auc=%.3f" % (recall_score(yte, yhat), precision_score(yte, yhat), roc_auc_score(yte, p)))

## 7. Choosing a decision threshold

In [ ]:
prec, rec, thr = precision_recall_curve(yte, p)
plt.figure(figsize=(6, 4)); plt.plot(thr, prec[:-1], label="precision"); plt.plot(thr, rec[:-1], label="recall")
plt.axhline(cfg["quality_gate"]["minimum_value"], color="red", ls=":", label="recall gate")
plt.xlabel("probability threshold"); plt.legend(); plt.title("Precision / recall trade-off"); plt.tight_layout()
plt.savefig("artifacts/threshold_tradeoff.png", dpi=120); plt.show()
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    yt = (p >= t).astype(int)
    print(f"threshold {t:.1f}: recall={recall_score(yte, yt):.3f}  precision={precision_score(yte, yt):.3f}  flagged={yt.mean():.1%}")

## 8. What drives predictions?

In [ ]:
imp = pd.Series(rf.feature_importances_, index=names).sort_values(ascending=False)
imp.index = [n.replace("num__", "").replace("cat__", "") for n in imp.index]
imp.head(15).iloc[::-1].plot.barh(figsize=(7, 5), title="Top 15 feature importances (Random Forest)")
plt.tight_layout(); plt.savefig("artifacts/feature_importance.png", dpi=120); plt.show()

## 9. Conclusions

- **Audio features and genre carry real but limited signal** (ROC-AUC 0.81-0.85 depending on the model). Genre dominates the importances (e.g. pop-film, chill, k-pop). Popularity also depends on things we do not observe: artist fame, marketing, playlists, release date.
- **The template's RandomForest is not the best model here**: HistGradientBoosting (ROC-AUC 0.85, recall 0.81) and even Logistic Regression (0.83) beat it. The MLOps pipeline keeps RandomForest to stay faithful to the template; swapping the estimator in `05_train_registry.py` / `12_retrain_model.py` is the obvious next improvement.
- At the default threshold the model **catches ~77% of hits** but with **low precision (~42%)**: most flagged tracks are not hits. It is a *screening* tool, not a verdict.
- The 0.70 recall quality gate from the churn template is met with some margin; raise the threshold (section 7) if false alarms are costly.
- Because the data is a static snapshot, the drift/retraining labs use *simulated* market shifts (`spotify_drift.py`) rather than real new data.